# Часть 3 — авторизация: в MCP-сервер ходят только свои

**stdio** защищать не нужно: сервер — дочерний процесс клиента, снаружи до него не добраться.

**HTTP** — другое дело: сервер слушает порт, и кто дотянется до адреса, тот и вызовет `run_sql`.
Нужна авторизация. В MCP она устроена как в обычных веб-API — заголовок

```
Authorization: Bearer <токен>
```

| способ | как | когда |
|---|---|---|
| **токен (API-ключ)** | у клиента есть секретная строка, сервер её сверяет | свои сервисы, внутренняя сеть — **делаем сейчас** |
| **OAuth 2.1** | токен выдаёт сервер авторизации (Keycloak, Auth0, Google…) после входа пользователя | публичные серверы, много пользователей; так работают GitHub, Notion и др. |

Для сервера оба способа выглядят одинаково: пришёл запрос с токеном → проверить → пустить или вернуть
**401**. В Python SDK MCP за проверку отвечает класс `TokenVerifier` — его и напишем.

## Шаг 1. Сервер с проверкой токена

Инструменты — те же, что в части 1 (импортируем их из `db_server.py`). Новое — только «замок»:
`CheckToken` сверяет токен из запроса с секретом. Плюс `AuthSettings` — по ним сервер
отвечает клиентам на 401 стандартным образом.

In [1]:
%%writefile secure_server.py
"""The bank tools over HTTP — only for clients that send the right token."""

import os
import secrets

from mcp.server.auth.provider import AccessToken, TokenVerifier
from mcp.server.auth.settings import AuthSettings
from mcp.server.fastmcp import FastMCP

from db_server import describe_table, list_tables, run_sql   # the tools from part 1, unchanged

TOKEN = os.environ["MCP_TOKEN"]                               # the secret, given at start
URL = "http://localhost:8012"


class CheckToken(TokenVerifier):
    """Called on every request with the token from the header `Authorization: Bearer <token>`."""

    async def verify_token(self, token: str) -> AccessToken | None:
        if secrets.compare_digest(token, TOKEN):              # constant-time compare
            return AccessToken(token=token, client_id="student", scopes=[])
        return None                                           # → 401 Unauthorized


mcp = FastMCP(
    "bank-secure", port=8012, log_level="WARNING",
    token_verifier=CheckToken(),
    auth=AuthSettings(issuer_url=URL, resource_server_url=f"{URL}/mcp", required_scopes=[],
                      validate_token_resource=False),         # our verifier checks the token itself
)
mcp.tool()(list_tables)
mcp.tool()(describe_table)
mcp.tool()(run_sql)

if __name__ == "__main__":
    mcp.run(transport="streamable-http")                      # http://localhost:8012/mcp

Overwriting secure_server.py


## Шаг 2. Запускаем с секретом

Секрет генерируем случайный и передаём серверу через переменную окружения — **не в коде**.
В жизни он лежит в `.env` / секретах докера / менеджере секретов.

In [2]:
import os
import secrets
import subprocess
import sys
import time

TOKEN = secrets.token_urlsafe(16)        # a random secret, e.g. 'pX3v...'
server = subprocess.Popen([sys.executable, "secure_server.py"], env={**os.environ, "MCP_TOKEN": TOKEN})
time.sleep(3)
print("сервер слушает http://localhost:8012/mcp")

сервер слушает http://localhost:8012/mcp


## Шаг 3. Чужой: без токена и с неверным токеном

Сначала «сырой» HTTP-запрос — посмотреть, что отвечает сервер:

In [4]:
import httpx

URL = "http://localhost:8012/mcp"

for name, headers in [("без токена", {}), ("неверный токен", {"Authorization": "Bearer guess"})]:
    response = httpx.post(URL, json={}, headers=headers)
    print(f"{name:<15} → {response.status_code} {response.json()['error_description']}")

без токена      → 401 Authentication required
неверный токен  → 401 Authentication required


Теперь MCP-клиент без токена — даже список инструментов не получить:

In [5]:
from langchain_mcp_adapters.client import MultiServerMCPClient

stranger = MultiServerMCPClient({"bank": {"transport": "streamable_http", "url": URL}})
try:
    await stranger.get_tools()
except Exception as error:
    print("не пустили:", error.exceptions[0] if hasattr(error, "exceptions") else error)

не пустили: Client error '401 Unauthorized' for url 'http://localhost:8012/mcp'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401


## Шаг 4. Свой: передаём токен в заголовке

Единственное отличие — `headers` в конфиге клиента:

In [6]:
client = MultiServerMCPClient({
    "bank": {
        "transport": "streamable_http",
        "url": URL,
        "headers": {"Authorization": f"Bearer {TOKEN}"},
    },
})

tools = await client.get_tools()
print([tool.name for tool in tools])

run_sql = next(tool for tool in tools if tool.name == "run_sql")
print((await run_sql.ainvoke({"sql": "SELECT count(*) AS loans FROM loans"}))[0]["text"])

['list_tables', 'describe_table', 'run_sql']
loans
682


## Шаг 5. Агент с токеном

Агенту токен не нужен вовсе — его передаёт MCP-клиент, а агент получает уже готовые инструменты.
Граф — как в части 2; здесь соберём его одной строкой `create_agent` (из занятия 1, это тот же
граф LangGraph «агент ⇄ инструменты»):

In [7]:
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

load_dotenv()
llm = ChatOpenAI(model=os.getenv("MODEL", "openai/gpt-4o-mini"), base_url="https://openrouter.ai/api/v1",
                 api_key=os.getenv("OPENROUTER_API_KEY"))

agent = create_agent(llm, tools, system_prompt="Ты аналитик банка. Отвечай по-русски, кратко, с цифрами. "
                                               "Сначала посмотри таблицы (list_tables), потом пиши SQL.")

result = await agent.ainvoke({"messages": [("user", "Какая средняя сумма кредита?")]})
for message in result["messages"]:
    if message.type == "ai" and message.tool_calls:
        print("🤖 →", [(call["name"], call["args"]) for call in message.tool_calls])
print("\n💬", result["messages"][-1].text)

🤖 → [('list_tables', {})]
🤖 → [('describe_table', {'table': 'loans'})]
🤖 → [('run_sql', {'sql': 'SELECT AVG(amount) AS average_loan_amount_czk, COUNT(*) AS loan_count FROM loans;'})]

💬 Средняя сумма кредита — **151 410,18 CZK** по **682 кредитам**.


In [9]:
server.terminate()

## Итоги

- stdio не нуждается в авторизации; HTTP-сервер — нуждается;
- клиент шлёт `Authorization: Bearer <токен>` — в `MultiServerMCPClient` это поле `headers`;
- сервер проверяет токен в `TokenVerifier`: верный → пускаем, нет → **401**;
- секрет — в переменных окружения, не в коде; в проде ещё **HTTPS** (иначе токен видно в сети)
  и свой токен на каждого клиента, чтобы любой можно было отозвать;
- для публичных серверов с пользователями — OAuth 2.1: тот же `TokenVerifier`, только он проверяет
  токены, выданные сервером авторизации.

Дальше — часть 4: подключаем чужие, готовые MCP-серверы.